# EDA — Exportaciones de Comercio Exterior (INE Bolivia)

Análisis exploratorio de **un archivo individual** de exportaciones del INE.

**Uso:** Este notebook recibe un parámetro `FILE_PATH` con la ruta al archivo `.xlsx`.
Se puede ejecutar directamente o mediante el notebook orquestador con `papermill`.

In [ ]:
# Parámetro de entrada — ruta al archivo de exportaciones
# Esta celda es inyectada por papermill cuando se ejecuta desde el orquestador.
FILE_PATH = r"data/raw/comercio exterior/exportaciones/EXPORTACIONES 2021.xlsx"

## 1. Configuración e Importaciones

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Asegurar que src/ es importable resolviendo la raíz del proyecto
current = Path.cwd().resolve()
candidates = [
    current,
    current.parent,
    current / "insight-bolivia",
    current.parent / "insight-bolivia",
    *current.parents,
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src" / "extract.py").exists()),
    current,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.extract import get_excel_metadata, read_ine_excel
from src.transform import (
    clean_export_dataframe,
    compute_null_report,
    parse_flujo,
)
from src.validate import run_export_validations

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

filepath = Path(FILE_PATH)
if not filepath.is_absolute():
    filepath = PROJECT_ROOT / filepath
print(f"Archivo: {filepath.name}")
print(f"Ruta completa: {filepath.resolve()}")

## 2. Metadatos del Archivo

In [ ]:
meta = get_excel_metadata(filepath)

print(f"Archivo:       {meta['filename']}")
print(f"Tamaño:        {meta['file_size_mb']} MB")
print(f"Hojas:         {meta['sheet_names']}")
print(f"Hoja activa:   {meta['active_sheet']}")
print(f"Columnas ({meta['n_columns']}):")
for i, col in enumerate(meta['headers'], 1):
    print(f"  {i:2d}. {col}")

## 3. Carga y Limpieza de Datos

In [ ]:
# Leer archivo completo (dtype=str para preservar ceros en NANDINA)
df_raw = read_ine_excel(filepath)
print(f"Shape crudo: {df_raw.shape}")
print(f"Columnas: {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:
# Aplicar pipeline de limpieza completo
df = clean_export_dataframe(df_raw)
print(f"Shape limpio: {df.shape}")
print(f"Columnas normalizadas: {list(df.columns)}")
df.dtypes

## 4. Análisis del Código NANDINA

In [ ]:
nandina = df["NANDINA"]
print(f"Tipo de dato predominante: {nandina.dtype}")
print(f"Valores únicos: {nandina.nunique()}")
print("Longitud (value_counts):")
print(nandina.str.len().value_counts().sort_index())
print(f"\nRegistros con cero a la izquierda: {(nandina.str[0] == '0').sum()}")
print(f"Porcentaje: {(nandina.str[0] == '0').mean() * 100:.1f}%")
print("\nMuestras con cero a la izquierda:")
print(nandina[nandina.str[0] == '0'].head(10).tolist())

## 5. Análisis del Campo FLUJO

In [ ]:
if "FLUJO" in df.columns:
    print("Valores únicos de FLUJO:")
    print(df["FLUJO"].value_counts())
    print()
    flujo_parsed = parse_flujo(df["FLUJO"])
    print("FLUJO parseado:")
    print(flujo_parsed.drop_duplicates())

## 6. Estadísticas de Valores y Pesos

In [ ]:
numeric_cols = ["VALOR", "KILBRU", "KILNET", "FINO"]
existing = [c for c in numeric_cols if c in df.columns]
print("Estadísticas descriptivas de columnas numéricas:")
df[existing].describe().round(2)

In [ ]:
# Verificar coherencia: KILBRU >= KILNET
if "KILBRU" in df.columns and "KILNET" in df.columns:
    mask = df["KILBRU"].notna() & df["KILNET"].notna()
    violations = df.loc[mask, "KILBRU"] < df.loc[mask, "KILNET"]
    print(f"Registros donde KILBRU < KILNET: {violations.sum()} de {mask.sum()}")
    if violations.sum() > 0:
        print("Muestras de violaciones:")
        print(df.loc[violations[violations].index, ["NANDINA", "DESNAN", "KILBRU", "KILNET"]].head())

## 7. Reporte de Nulos

In [ ]:
null_report = compute_null_report(df)
print("Reporte de nulos (ordenado por % descendente):")
# Mostrar solo columnas con nulos > 0
with_nulls = null_report[null_report["nulos"] > 0]
if with_nulls.empty:
    print("¡Sin nulos!")
else:
    print(with_nulls.to_string(index=False))

## 8. Validaciones de Calidad

In [ ]:
results = run_export_validations(df)
print("Resultados de validación:")
print("-" * 60)
for r in results:
    status = "✅ PASS" if r.passed else "❌ FAIL"
    print(f"{status} | {r.rule_name}: {r.message}")
    if r.details:
        for k, v in r.details.items():
            print(f"         {k}: {v}")
    print()

## 9. Distribución por País y Departamento

In [ ]:
if "DESPAIS" in df.columns:
    print("Top 15 países destino (por número de registros):")
    print(df["DESPAIS"].value_counts().head(15))

if "DESDEP" in df.columns:
    print("\nRegistros por departamento de origen:")
    print(df["DESDEP"].value_counts())

## 10. Resumen del Archivo

In [ ]:
print("=" * 60)
print(f"RESUMEN: {filepath.name}")
print("=" * 60)
print(f"  Filas:              {len(df):,}")
print(f"  Columnas:           {len(df.columns)}")
if "GESTION" in df.columns:
    print(f"  Gestión(es):        {sorted(df['GESTION'].dropna().unique())}")
if "MES" in df.columns:
    print(f"  Meses:              {sorted(df['MES'].dropna().unique())}")
if "NANDINA" in df.columns:
    print(f"  Productos únicos:   {df['NANDINA'].nunique():,}")
if "DESPAIS" in df.columns:
    print(f"  Países destino:     {df['DESPAIS'].nunique()}")
if "VALOR" in df.columns:
    print(f"  Valor FOB total:    USD {df['VALOR'].sum():,.2f}")
print(f"  Validaciones:       {sum(1 for r in results if r.passed)}/{len(results)} pasaron")